In [2]:
import os 
os.environ["HTTP_PROXY"] = "http://proxy.cat.com:80"
os.environ["HTTPS_PROXY"] = "http://proxy.cat.com:80"
os.environ["NO_PROXY"] = "localhost, .cat.com, 169.254.169.254, *.amazonaws.com, .us-east-2.privatelink.snowflakecomputing.com"

In [5]:
!ls

LICENSE                      results
MANIFEST.in                  run.py
README.md                    setup.py
auto_error_identification.py tau_bench
few_shot_data                tau_bench.egg-info
historical_trajectories      translate.ipynb


In [50]:
import json
import random

# Load the original JSON file
file_path = "./tau_bench/envs/airline/data/users.json"
with open(file_path, "r") as f:
    users_data = json.load(f)

# Kazakh and Russian names for adaptation
kazakh_first_names_male = ["Алихан", "Даулет", "Данияр", "Мадияр", "Ерасыл", "Бексултан", "Нуржан", "Даир", "Рустам", "Бекжан", "Айдос", "Азамат", "Максат"]
kazakh_first_names_female = ["Жанар", "Айгуль", "Мадина", "Гульмира", "Айсулу", "Айша", "Алина", "Дана", "Диана", "Сабина", "Аяна", "Айгерим", "Ботагоз", "Ажар"]
kazakh_last_names_male = ["Тлеубеков", "Сагиндыков", "Нуртазин", "Кайратов", "Ергазы", "Айдосов", "Касымов", "Жумагалиев", "Мустафин", "Канатов", "Алиев"]
kazakh_last_names_female = ["Тлеубекова", "Сагиндыкова", "Нуртазина", "Кайратова", "Ергазы", "Айдосова", "Касымова", "Жумагалиева", "Мустафина", "Канатова", "Алиева"]
russian_first_names = ["Алексей", "Дмитрий", "Владимир",  "Павел", "Сергей", ]
russian_last_names = ["Иванов", "Смирнов", "Кузнецов", "Попов", "Соловьев", "Михайлов", "Новиков", "Федоров", "Борисов", "Васильев"]

# Kazakh addresses (authentic street names and cities)
kazakh_addresses = [
    {"address1": "проспект Абая 58", "city": "Алматы", "province": "Алматинская", "zip": "050000"},
    {"address1": "улица Назарбаева 15", "city": "Астана", "province": "Акмолинская", "zip": "010000"},
    {"address1": "проспект Достык 101", "city": "Шымкент", "province": "Туркестанская", "zip": "160000"},
    {"address1": "улица Абылай хана 20", "city": "Караганда", "province": "Карагандинская", "zip": "100000"},
    {"address1": "проспект Республики 75", "city": "Костанай", "province": "Костанайская", "zip": "110000"},
    {"address1": "проспект Абая 20", "city": "Алматы", "province": "Алматинская", "zip": "050000"},
    {"address1": "улица Назарбаева 9", "city": "Астана", "province": "Акмолинская", "zip": "010000"},
    {"address1": "проспект Достык 21", "city": "Шымкент", "province": "Туркестанская", "zip": "160000"},
    {"address1": "улица Абылай хана 99", "city": "Караганда", "province": "Карагандинская", "zip": "100000"},
    {"address1": "проспект Республики 5", "city": "Костанай", "province": "Костанайская", "zip": "110000"},
    {"address1": "проспект Толе би 30", "city": "Тараз", "province": "Жамбылская", "zip": "080000"},
    {"address1": "улица Евразия 45", "city": "Уральск", "province": "Западно-Казахстанская", "zip": "090000"},
    {"address1": "проспект Победы 12", "city": "Усть-Каменогорск", "province": "Восточно-Казахстанская", "zip": "070000"},
    {"address1": "проспект Мангилик Ел 25", "city": "Атырау", "province": "Атырауская", "zip": "060000"},
    {"address1": "улица Айманова 7", "city": "Павлодар", "province": "Павлодарская", "zip": "140000"},
    {"address1": "улица Сейфуллина 55", "city": "Кызылорда", "province": "Кызылординская", "zip": "120000"},
    {"address1": "проспект Назарбаева 99", "city": "Актау", "province": "Мангистауская", "zip": "130000"},
    {"address1": "улица Гоголя 18", "city": "Семей", "province": "Восточно-Казахстанская", "zip": "071400"},
    {"address1": "проспект Бейбитшилик 40", "city": "Петропавловск", "province": "Северо-Казахстанская", "zip": "150000"},
    {"address1": "улица Дружбы Народов 28", "city": "Актобе", "province": "Актюбинская", "zip": "030000"},
    {"address1": "проспект Байтурсынова 15", "city": "Кокшетау", "province": "Акмолинская", "zip": "020000"},
    {"address1": "улица Желтоксан 23", "city": "Туркестан", "province": "Туркестанская", "zip": "161200"},
    {"address1": "проспект Богенбай батыра 32", "city": "Темиртау", "province": "Карагандинская", "zip": "101400"},
    {"address1": "улица Габдуллина 9", "city": "Жезказган", "province": "Улытауская", "zip": "100600"},
    {"address1": "улица Акмешит 55", "city": "Экибастуз", "province": "Павлодарская", "zip": "141200"}
]
from unidecode import unidecode

def cyrillic_to_latin(text):
    return unidecode(text)

def get_random_passenger():
    gender = random.choice(["male", "female"])  # Randomly assign gender
    if gender == "male":
        first_name = random.choice(kazakh_first_names_male + russian_first_names)
        last_name = random.choice(kazakh_last_names_male + russian_last_names)
    else:
        first_name = random.choice(kazakh_first_names_female)
        last_name = random.choice(kazakh_last_names_female)
    
    return first_name, last_name

# Function to generate a new user ID (English format)
def generate_user_id(name, old_id):
    
    first_name = cyrillic_to_latin(name["first_name"]).lower()
    last_name = cyrillic_to_latin(name["last_name"]).lower()
    return f"{first_name}_{last_name}_{old_id.split('_')[-1]}"

# Function to generate an English email format
def generate_email(name):
    first_name = cyrillic_to_latin(name["first_name"]).lower()
    last_name = cyrillic_to_latin(name["last_name"]).lower()
    number = random.randint(1000, 9999)
    return f"{first_name}.{last_name}{number}@example.com"

# Create the adapted dataset
adapted_users = {}
for user_id, user_info in users_data.items():
    # Assign Kazakh or Russian names randomly
    if random.random() < 0.8:  # 80% chance of being Kazakh
        if random.random() < 0.5:
            new_name = {
                "first_name": random.choice(kazakh_first_names_male),
                "last_name": random.choice(kazakh_last_names_male),
            }
        else:
            new_name = {
                "first_name": random.choice(kazakh_first_names_female),
                "last_name": random.choice(kazakh_last_names_female),
            }
    else:
        new_name = {
            "first_name": random.choice(russian_first_names),
            "last_name": random.choice(russian_last_names),
        }

    # Assign a new Kazakh address
    new_address = random.choice(kazakh_addresses)
    new_address.update({"country": "Казахстан"})

    # Generate a new user ID and email
    new_user_id = generate_user_id(new_name, user_id)
    new_email = generate_email(new_name)

    

    # Update saved passengers with Kazakh or Russian names
    updated_passengers = []
    for p in user_info.get("saved_passengers", []):
        rand_passenger = get_random_passenger()
        updated_passengers.append(
            {
            "first_name": rand_passenger[0],
            "last_name": rand_passenger[1],
            "dob": p["dob"],
        }
        )


    # Construct the new user entry
    adapted_users[new_user_id] = {
        "name": new_name,
        "address": new_address,
        "email": new_email,
        "dob": user_info["dob"],
        "payment_methods": user_info["payment_methods"],
        "saved_passengers": updated_passengers,
        "membership": user_info["membership"],
        "reservations": user_info["reservations"],
    }

# Save the modified dataset as JSON
output_file_path = "./tau_bench/envs/airline/data/users_kk.json"
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(adapted_users, f, ensure_ascii=False, indent=4)




In [51]:
# Load the original JSON file
file_path = "./tau_bench/envs/airline/data/flights.json"
with open(file_path, "r") as f:
    flights_data = json.load(f)

In [52]:
flights_data

{'HAT001': {'flight_number': 'HAT001',
  'origin': 'PHL',
  'destination': 'LGA',
  'scheduled_departure_time_est': '06:00:00',
  'scheduled_arrival_time_est': '07:00:00',
  'dates': {'2024-05-01': {'status': 'landed',
    'actual_departure_time_est': '2024-05-01T06:26:00',
    'actual_arrival_time_est': '2024-05-01T06:58:00'},
   '2024-05-02': {'status': 'landed',
    'actual_departure_time_est': '2024-05-02T05:35:00',
    'actual_arrival_time_est': '2024-05-02T06:51:00'},
   '2024-05-03': {'status': 'cancelled'},
   '2024-05-04': {'status': 'landed',
    'actual_departure_time_est': '2024-05-04T06:06:00',
    'actual_arrival_time_est': '2024-05-04T06:59:00'},
   '2024-05-05': {'status': 'landed',
    'actual_departure_time_est': '2024-05-05T05:34:00',
    'actual_arrival_time_est': '2024-05-05T06:49:00'},
   '2024-05-06': {'status': 'landed',
    'actual_departure_time_est': '2024-05-06T05:53:00',
    'actual_arrival_time_est': '2024-05-06T07:00:00'},
   '2024-05-07': {'status': 'lan

In [53]:
city_codes = []
for  flight_number, flight_info in flights_data.items():
    city_codes.extend([flight_info["origin"], flight_info["destination"]])


In [54]:
set(city_codes)

{'ATL',
 'BOS',
 'CLT',
 'DEN',
 'DFW',
 'DTW',
 'EWR',
 'IAH',
 'JFK',
 'LAS',
 'LAX',
 'LGA',
 'MCO',
 'MIA',
 'MSP',
 'ORD',
 'PHL',
 'PHX',
 'SEA',
 'SFO'}

In [55]:
city_codes_mapping = {
    "ATL": "CIT",
    "BOS": "UKK",
    "CLT": "DMB",
    "DEN": "PWQ",
    "DFW": "KSN",
    "DTW": "PPK",
    "EWR": "NQZ",
    "IAH": "SCO",
    "JFK": "ALA",
    "LAS": "KZO",
    "LAX": "GUW",
    "LGA": "ALA",
    "MCO": "URA",
    "MIA": "AKX",
    "MSP": "PPK",
    "ORD": "NQZ",
    "PHL": "UKK",
    "PHX": "KZO",
    "SEA": "PPK",
    "SFO": "ALA"
}


In [56]:
adapted_flights = {}
for flight_number, flight_info in flights_data.items():
    adapted_flights[flight_number] = {
        "flight_number": flight_number,
        "origin": city_codes_mapping[flight_info["origin"]],
        "destination": city_codes_mapping[flight_info["destination"]],
        "scheduled_departure_time_est": flight_info["scheduled_departure_time_est"],
        "scheduled_arrival_time_est": flight_info["scheduled_arrival_time_est"],
        "dates": flight_info["dates"]
    }

In [57]:
adapted_flights

{'HAT001': {'flight_number': 'HAT001',
  'origin': 'UKK',
  'destination': 'ALA',
  'scheduled_departure_time_est': '06:00:00',
  'scheduled_arrival_time_est': '07:00:00',
  'dates': {'2024-05-01': {'status': 'landed',
    'actual_departure_time_est': '2024-05-01T06:26:00',
    'actual_arrival_time_est': '2024-05-01T06:58:00'},
   '2024-05-02': {'status': 'landed',
    'actual_departure_time_est': '2024-05-02T05:35:00',
    'actual_arrival_time_est': '2024-05-02T06:51:00'},
   '2024-05-03': {'status': 'cancelled'},
   '2024-05-04': {'status': 'landed',
    'actual_departure_time_est': '2024-05-04T06:06:00',
    'actual_arrival_time_est': '2024-05-04T06:59:00'},
   '2024-05-05': {'status': 'landed',
    'actual_departure_time_est': '2024-05-05T05:34:00',
    'actual_arrival_time_est': '2024-05-05T06:49:00'},
   '2024-05-06': {'status': 'landed',
    'actual_departure_time_est': '2024-05-06T05:53:00',
    'actual_arrival_time_est': '2024-05-06T07:00:00'},
   '2024-05-07': {'status': 'lan

In [58]:
output_file_path = "./tau_bench/envs/airline/data/flights_kk.json"
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(adapted_flights, f, ensure_ascii=False, indent=4)

In [59]:
# Load the original JSON file
file_path = "./tau_bench/envs/airline/data/reservations.json"
with open(file_path, "r") as f:
    reservations_data = json.load(f)

In [60]:
reservations_data

{'4WQ150': {'reservation_id': '4WQ150',
  'user_id': 'chen_jackson_3290',
  'origin': 'DFW',
  'destination': 'LAX',
  'flight_type': 'round_trip',
  'cabin': 'business',
  'flights': [{'origin': 'DFW',
    'destination': 'LAX',
    'flight_number': 'HAT170',
    'date': '2024-05-22',
    'price': 883},
   {'origin': 'LAX',
    'destination': 'DFW',
    'flight_number': 'HAT022',
    'date': '2024-05-26',
    'price': 779}],
  'passengers': [{'first_name': 'Chen',
    'last_name': 'Jackson',
    'dob': '1956-07-07'},
   {'first_name': 'Raj', 'last_name': 'Smith', 'dob': '1967-04-01'},
   {'first_name': 'Fatima', 'last_name': 'Martin', 'dob': '1970-01-20'}],
  'payment_history': [{'payment_id': 'gift_card_3576581', 'amount': 4986}],
  'created_at': '2024-05-02T03:10:19',
  'total_baggages': 5,
  'nonfree_baggages': 0,
  'insurance': 'no'},
 'VAAOXJ': {'reservation_id': 'VAAOXJ',
  'user_id': 'lei_rossi_3206',
  'origin': 'CLT',
  'destination': 'MCO',
  'flight_type': 'one_way',
  'cabi

In [119]:
adapted_users

{'aiana_ergazy_3668': {'name': {'first_name': 'Аяна', 'last_name': 'Ергазы'},
  'address': {'address1': 'улица Дружбы Народов 28',
   'city': 'Актобе',
   'province': 'Актюбинская',
   'zip': '030000',
   'country': 'Казахстан'},
  'email': 'aiana.ergazy6957@example.com',
  'dob': '1990-04-05',
  'payment_methods': {'credit_card_4421486': {'source': 'credit_card',
    'brand': 'visa',
    'last_four': '7447',
    'id': 'credit_card_4421486'},
   'certificate_4856383': {'source': 'certificate',
    'amount': 100,
    'id': 'certificate_4856383'},
   'certificate_7504069': {'source': 'certificate',
    'amount': 250,
    'id': 'certificate_7504069'},
   'credit_card_1955700': {'source': 'credit_card',
    'brand': 'visa',
    'last_four': '1907',
    'id': 'credit_card_1955700'}},
  'saved_passengers': [{'first_name': 'Жанар',
    'last_name': 'Мустафина',
    'dob': '1957-03-21'}],
  'membership': 'gold',
  'reservations': ['NO6JO3', 'AIXC49', 'HKEG34']},
 'bekzhan_ergazy_8984': {'name'

In [61]:
ids_mapping = {}
for id_original, id_adapted in zip(users_data.keys(), adapted_users.keys()):
    ids_mapping[id_original] = id_adapted


In [121]:
for info in adapted_users.values():
    print(info)

{'name': {'first_name': 'Аяна', 'last_name': 'Ергазы'}, 'address': {'address1': 'улица Дружбы Народов 28', 'city': 'Актобе', 'province': 'Актюбинская', 'zip': '030000', 'country': 'Казахстан'}, 'email': 'aiana.ergazy6957@example.com', 'dob': '1990-04-05', 'payment_methods': {'credit_card_4421486': {'source': 'credit_card', 'brand': 'visa', 'last_four': '7447', 'id': 'credit_card_4421486'}, 'certificate_4856383': {'source': 'certificate', 'amount': 100, 'id': 'certificate_4856383'}, 'certificate_7504069': {'source': 'certificate', 'amount': 250, 'id': 'certificate_7504069'}, 'credit_card_1955700': {'source': 'credit_card', 'brand': 'visa', 'last_four': '1907', 'id': 'credit_card_1955700'}}, 'saved_passengers': [{'first_name': 'Жанар', 'last_name': 'Мустафина', 'dob': '1957-03-21'}], 'membership': 'gold', 'reservations': ['NO6JO3', 'AIXC49', 'HKEG34']}
{'name': {'first_name': 'Бекжан', 'last_name': 'Ергазы'}, 'address': {'address1': 'проспект Мангилик Ел 25', 'city': 'Атырау', 'province'

In [125]:
for s in zip(users_data.values(), adapted_users.values()):
    print(s)

({'name': {'first_name': 'Mia', 'last_name': 'Li'}, 'address': {'address1': '975 Sunset Drive', 'address2': 'Suite 217', 'city': 'Austin', 'country': 'USA', 'province': 'TX', 'zip': '78750'}, 'email': 'mia.li3818@example.com', 'dob': '1990-04-05', 'payment_methods': {'credit_card_4421486': {'source': 'credit_card', 'brand': 'visa', 'last_four': '7447', 'id': 'credit_card_4421486'}, 'certificate_4856383': {'source': 'certificate', 'amount': 100, 'id': 'certificate_4856383'}, 'certificate_7504069': {'source': 'certificate', 'amount': 250, 'id': 'certificate_7504069'}, 'credit_card_1955700': {'source': 'credit_card', 'brand': 'visa', 'last_four': '1907', 'id': 'credit_card_1955700'}}, 'saved_passengers': [{'first_name': 'Amelia', 'last_name': 'Ahmed', 'dob': '1957-03-21'}], 'membership': 'gold', 'reservations': ['NO6JO3', 'AIXC49', 'HKEG34']}, {'name': {'first_name': 'Аяна', 'last_name': 'Ергазы'}, 'address': {'address1': 'улица Дружбы Народов 28', 'city': 'Актобе', 'province': 'Актюбинск

In [126]:
user_info_mapping = {}
for info_orginial, info_adapted in zip(users_data.values(), adapted_users.values()):
    user_info_mapping[(info_orginial["name"]["first_name"], info_orginial["name"]["last_name"])] = info_adapted["name"]["first_name"], info_adapted["name"]["last_name"]

In [169]:
user_info_mapping

{('mia_li_3668', 'Mia', 'Li'): ('aiana_ergazy_3668', 'Аяна', 'Ергазы'),
 ('mei_hernandez_8984', 'Mei', 'Hernandez'): ('bekzhan_ergazy_8984',
  'Бекжан',
  'Ергазы'),
 ('aarav_nguyen_1055', 'Aarav', 'Nguyen'): ('vladimir_novikov_1055',
  'Владимир',
  'Новиков'),
 ('chen_hernandez_2608', 'Chen', 'Hernandez'): ('rustam_nurtazin_2608',
  'Рустам',
  'Нуртазин'),
 ('lucas_hernandez_8985', 'Lucas', 'Hernandez'): ('beksultan_kanatov_8985',
  'Бексултан',
  'Канатов'),
 ('sophia_taylor_9065', 'Sophia', 'Taylor'): ('erasyl_tleubekov_9065',
  'Ерасыл',
  'Тлеубеков'),
 ('liam_santos_5621', 'Liam', 'Santos'): ('erasyl_aidosov_5621',
  'Ерасыл',
  'Айдосов'),
 ('olivia_smith_4705', 'Olivia', 'Smith'): ('aleksei_ivanov_4705',
  'Алексей',
  'Иванов'),
 ('sophia_davis_8874', 'Sophia', 'Davis'): ('madina_mustafina_8874',
  'Мадина',
  'Мустафина'),
 ('ava_davis_9130', 'Ava', 'Davis'): ("aigul'_kairatova_9130",
  'Айгуль',
  'Кайратова'),
 ('mia_jackson_2156', 'Mia', 'Jackson'): ('rustam_zhumagaliev_

In [159]:
user_info_mapping = {}

In [165]:
user_info_mapping[('Mia', 'Li')]

('Алексей', 'Кузнецов')

In [167]:
user_info_mapping = {}
for user_id, info_original in users_data.items():
    adapted_user_name = adapted_users[ids_mapping[user_id]]["name"]
    user_info_mapping[(user_id, info_original["name"]["first_name"], info_original["name"]["last_name"])] = ids_mapping[user_id], adapted_user_name["first_name"], adapted_user_name["last_name"]
    print(user_info_mapping[(user_id, info_original["name"]["first_name"], info_original["name"]["last_name"])] )

('aiana_ergazy_3668', 'Аяна', 'Ергазы')
('bekzhan_ergazy_8984', 'Бекжан', 'Ергазы')
('vladimir_novikov_1055', 'Владимир', 'Новиков')
('rustam_nurtazin_2608', 'Рустам', 'Нуртазин')
('beksultan_kanatov_8985', 'Бексултан', 'Канатов')
('erasyl_tleubekov_9065', 'Ерасыл', 'Тлеубеков')
('erasyl_aidosov_5621', 'Ерасыл', 'Айдосов')
('aleksei_ivanov_4705', 'Алексей', 'Иванов')
('madina_mustafina_8874', 'Мадина', 'Мустафина')
("aigul'_kairatova_9130", 'Айгуль', 'Кайратова')
('rustam_zhumagaliev_2156', 'Рустам', 'Жумагалиев')
('aisha_mustafina_3449', 'Айша', 'Мустафина')
('dana_kasymova_9431', 'Дана', 'Касымова')
('bekzhan_tleubekov_1929', 'Бекжан', 'Тлеубеков')
('pavel_mikhailov_2879', 'Павел', 'Михайлов')
('nurzhan_zhumagaliev_4690', 'Нуржан', 'Жумагалиев')
('azamat_tleubekov_4874', 'Азамат', 'Тлеубеков')
("aleksei_solov'ev_3082", 'Алексей', 'Соловьев')
("gul'mira_kairatova_3653", 'Гульмира', 'Кайратова')
('madiiar_aidosov_4633', 'Мадияр', 'Айдосов')
('diana_alieva_1853', 'Диана', 'Алиева')
('da

In [150]:
users_data["mia_li_3668"]

{'name': {'first_name': 'Mia', 'last_name': 'Li'},
 'address': {'address1': '975 Sunset Drive',
  'address2': 'Suite 217',
  'city': 'Austin',
  'country': 'USA',
  'province': 'TX',
  'zip': '78750'},
 'email': 'mia.li3818@example.com',
 'dob': '1990-04-05',
 'payment_methods': {'credit_card_4421486': {'source': 'credit_card',
   'brand': 'visa',
   'last_four': '7447',
   'id': 'credit_card_4421486'},
  'certificate_4856383': {'source': 'certificate',
   'amount': 100,
   'id': 'certificate_4856383'},
  'certificate_7504069': {'source': 'certificate',
   'amount': 250,
   'id': 'certificate_7504069'},
  'credit_card_1955700': {'source': 'credit_card',
   'brand': 'visa',
   'last_four': '1907',
   'id': 'credit_card_1955700'}},
 'saved_passengers': [{'first_name': 'Amelia',
   'last_name': 'Ahmed',
   'dob': '1957-03-21'}],
 'membership': 'gold',
 'reservations': ['NO6JO3', 'AIXC49', 'HKEG34']}

In [145]:
adapted_users["aiana_ergazy_3668"]["name"]["first_name"]

'Аяна'

In [142]:
ids_mapping["mia_li_3668"]

'aiana_ergazy_3668'

In [152]:
user_info_mapping

{('Mia', 'Li'): ('Алексей', 'Кузнецов'),
 ('Mei', 'Hernandez'): ('Ерасыл', 'Канатов'),
 ('Aarav', 'Nguyen'): ('Дана', 'Айдосова'),
 ('Chen', 'Hernandez'): ('Рустам', 'Нуртазин'),
 ('Lucas', 'Hernandez'): ('Алина', 'Канатова'),
 ('Sophia', 'Taylor'): ('Ерасыл', 'Тлеубеков'),
 ('Liam', 'Santos'): ('Ерасыл', 'Айдосов'),
 ('Olivia', 'Smith'): ('Ботагоз', 'Ергазы'),
 ('Sophia', 'Davis'): ('Мадина', 'Мустафина'),
 ('Ava', 'Davis'): ('Айша', 'Касымова'),
 ('Mia', 'Jackson'): ('Рустам', 'Жумагалиев'),
 ('Liam', 'Taylor'): ('Айгуль', 'Касымова'),
 ('Emma', 'Nguyen'): ('Дана', 'Касымова'),
 ('Yara', 'Silva'): ('Даулет', 'Тлеубеков'),
 ('Aarav', 'Jackson'): ('Павел', 'Михайлов'),
 ('Raj', 'Garcia'): ('Дмитрий', 'Смирнов'),
 ('Lei', 'Rossi'): ('Павел', 'Михайлов'),
 ('Harper', 'Kovacs'): ('Алексей', 'Соловьев'),
 ('Isabella', 'Ito'): ('Алихан', 'Тлеубеков'),
 ('Isabella', 'Garcia'): ('Айдос', 'Айдосов'),
 ('Lucas', 'Sanchez'): ('Диана', 'Алиева'),
 ('Isabella', 'Khan'): ('Мадина', 'Алиева'),
 ('Am

In [96]:
user_info_mapping[("Mia", "Li")][0]

'Алексей'

In [88]:
users_data

{'mia_li_3668': {'name': {'first_name': 'Mia', 'last_name': 'Li'},
  'address': {'address1': '975 Sunset Drive',
   'address2': 'Suite 217',
   'city': 'Austin',
   'country': 'USA',
   'province': 'TX',
   'zip': '78750'},
  'email': 'mia.li3818@example.com',
  'dob': '1990-04-05',
  'payment_methods': {'credit_card_4421486': {'source': 'credit_card',
    'brand': 'visa',
    'last_four': '7447',
    'id': 'credit_card_4421486'},
   'certificate_4856383': {'source': 'certificate',
    'amount': 100,
    'id': 'certificate_4856383'},
   'certificate_7504069': {'source': 'certificate',
    'amount': 250,
    'id': 'certificate_7504069'},
   'credit_card_1955700': {'source': 'credit_card',
    'brand': 'visa',
    'last_four': '1907',
    'id': 'credit_card_1955700'}},
  'saved_passengers': [{'first_name': 'Amelia',
    'last_name': 'Ahmed',
    'dob': '1957-03-21'}],
  'membership': 'gold',
  'reservations': ['NO6JO3', 'AIXC49', 'HKEG34']},
 'mei_hernandez_8984': {'name': {'first_name':

In [87]:
ids_mapping

{'mia_li_3668': 'aiana_ergazy_3668',
 'mei_hernandez_8984': 'bekzhan_ergazy_8984',
 'aarav_nguyen_1055': 'vladimir_novikov_1055',
 'chen_hernandez_2608': 'rustam_nurtazin_2608',
 'lucas_hernandez_8985': 'beksultan_kanatov_8985',
 'sophia_taylor_9065': 'erasyl_tleubekov_9065',
 'liam_santos_5621': 'erasyl_aidosov_5621',
 'olivia_smith_4705': 'aleksei_ivanov_4705',
 'sophia_davis_8874': 'madina_mustafina_8874',
 'ava_davis_9130': "aigul'_kairatova_9130",
 'mia_jackson_2156': 'rustam_zhumagaliev_2156',
 'liam_taylor_3449': 'aisha_mustafina_3449',
 'emma_nguyen_9431': 'dana_kasymova_9431',
 'yara_silva_1929': 'bekzhan_tleubekov_1929',
 'aarav_jackson_2879': 'pavel_mikhailov_2879',
 'raj_garcia_4690': 'nurzhan_zhumagaliev_4690',
 'lei_rossi_4874': 'azamat_tleubekov_4874',
 'harper_kovacs_3082': "aleksei_solov'ev_3082",
 'isabella_ito_3653': "gul'mira_kairatova_3653",
 'isabella_garcia_4633': 'madiiar_aidosov_4633',
 'lucas_sanchez_1853': 'diana_alieva_1853',
 'isabella_khan_6576': 'dair_kan

In [62]:
[info['saved_passengers'] for info in users_data.values()]

[[{'first_name': 'Amelia', 'last_name': 'Ahmed', 'dob': '1957-03-21'}],
 [{'first_name': 'Anya', 'last_name': 'Anderson', 'dob': '1959-05-28'}],
 [{'first_name': 'Mohamed', 'last_name': 'Johnson', 'dob': '1981-07-16'},
  {'first_name': 'Olivia', 'last_name': 'Sanchez', 'dob': '1983-05-09'}],
 [{'first_name': 'Liam', 'last_name': 'Nguyen', 'dob': '1960-08-07'}],
 [{'first_name': 'Harper', 'last_name': 'Taylor', 'dob': '1986-02-14'}],
 [{'first_name': 'Lucas', 'last_name': 'Davis', 'dob': '1987-10-27'},
  {'first_name': 'Mason', 'last_name': 'Khan', 'dob': '1983-09-06'}],
 [{'first_name': 'Omar', 'last_name': 'Davis', 'dob': '1991-03-04'}],
 [{'first_name': 'Raj', 'last_name': 'Li', 'dob': '1991-10-28'},
  {'first_name': 'Anya', 'last_name': 'Gonzalez', 'dob': '1974-12-23'}],
 [{'first_name': 'Liam', 'last_name': 'Rossi', 'dob': '1995-08-06'},
  {'first_name': 'Aarav', 'last_name': 'Anderson', 'dob': '1955-03-26'}],
 [{'first_name': 'Emma', 'last_name': 'Gonzalez', 'dob': '2000-09-27'}],

In [63]:
# passengers_mapping = {}
# for info_original, info_adapted in zip(users_data.values(), adapted_users.values()):
#     passengers_mapping[info_original["saved_passengers"]] = info_adapted["saved_passengers"]

# passenger_mapping = {
#     tuple(tuple(p.items()) for p in orig_list): adapted_list
#     for orig_list, adapted_list in zip([info['saved_passengers'] for info in users_data.values()], [info['saved_passengers'] for info in adapted_users.values()])
# }

passenger_mapping = {
    (p['first_name'], p['last_name'], p['dob']): adapted_p
    for orig_list, adapted_list in zip([info['saved_passengers'] for info in users_data.values()], [info['saved_passengers'] for info in adapted_users.values()])
    for p, adapted_p in zip(orig_list, adapted_list)
}



In [64]:
passenger_mapping

{('Amelia', 'Ahmed', '1957-03-21'): {'first_name': 'Жанар',
  'last_name': 'Мустафина',
  'dob': '1957-03-21'},
 ('Anya', 'Anderson', '1959-05-28'): {'first_name': 'Данияр',
  'last_name': 'Кайратов',
  'dob': '1959-05-28'},
 ('Mohamed', 'Johnson', '1981-07-16'): {'first_name': 'Айдос',
  'last_name': 'Федоров',
  'dob': '1981-07-16'},
 ('Olivia', 'Sanchez', '1983-05-09'): {'first_name': 'Ерасыл',
  'last_name': 'Васильев',
  'dob': '1983-05-09'},
 ('Liam', 'Nguyen', '1960-08-07'): {'first_name': 'Максат',
  'last_name': 'Касымов',
  'dob': '1960-08-07'},
 ('Harper', 'Taylor', '1986-02-14'): {'first_name': 'Рустам',
  'last_name': 'Айдосов',
  'dob': '1986-02-14'},
 ('Lucas', 'Davis', '1987-10-27'): {'first_name': 'Мадина',
  'last_name': 'Тлеубекова',
  'dob': '1987-10-27'},
 ('Mason', 'Khan', '1983-09-06'): {'first_name': 'Ажар',
  'last_name': 'Касымова',
  'dob': '1983-09-06'},
 ('Omar', 'Davis', '1991-03-04'): {'first_name': 'Ботагоз',
  'last_name': 'Касымова',
  'dob': '1991-03-

In [65]:
original_entry = {'first_name': 'Amelia',
    'last_name': 'Ahmed',
    'dob': '1957-03-21'}

In [66]:
adapted_version = passenger_mapping.get((original_entry['first_name'], original_entry['last_name'], original_entry['dob']))

In [67]:
(rand_passenger := get_random_passenger())[0]

'Айдос'

In [68]:
def get_random_passenger_dict():
    gender = random.choice(["male", "female"])  # Randomly assign gender
    rand_passenger = {}
    if gender == "male":
        rand_passenger["first_name"] = random.choice(kazakh_first_names_male + russian_first_names)
        rand_passenger["last_name"]  = random.choice(kazakh_last_names_male + russian_last_names)
    else:
        rand_passenger["first_name"]  = random.choice(kazakh_first_names_female)
        rand_passenger["last_name"]  = random.choice(kazakh_last_names_female)
    
    return rand_passenger

In [69]:
adapted_reservations = {}
for reservation_id, reservation_info in reservations_data.items():
    adapted_reservations[reservation_id] = {
        "reservation_id": reservation_id,
        "user_id": ids_mapping[reservation_info["user_id"]],
        "origin": city_codes_mapping[reservation_info["origin"]],
        "destination": city_codes_mapping[reservation_info["destination"]],
        "flight_type": reservation_info["flight_type"],
        "cabin": reservation_info["cabin"],
        "flights": [
            {
                "origin": city_codes_mapping[flight["origin"]],
                "destination": city_codes_mapping[flight["destination"]],
                "flight_number": flight["flight_number"],
                "date": flight["date"],
                "price": flight["price"],
            }
            for flight in reservation_info["flights"]
        ],
        "passengers": [
            {
                "first_name": (passenger_mapping.get((passenger['first_name'], passenger['last_name'], passenger['dob'])) or (rand_passenger := get_random_passenger_dict()))["first_name"],
                "last_name": (passenger_mapping.get((passenger['first_name'], passenger['last_name'], passenger['dob'])) or rand_passenger)["last_name"],
                "dob": passenger['dob']
            }
            for passenger in reservation_info["passengers"]
        ],
        "payment_history": reservation_info["payment_history"],
        "created_at": reservation_info["created_at"],
        "total_baggages": reservation_info["total_baggages"],
        "nonfree_baggages": reservation_info["nonfree_baggages"],
        "insurance": reservation_info["insurance"],
    }

In [70]:
adapted_reservations

{'4WQ150': {'reservation_id': '4WQ150',
  'user_id': 'diana_mustafina_3290',
  'origin': 'KSN',
  'destination': 'GUW',
  'flight_type': 'round_trip',
  'cabin': 'business',
  'flights': [{'origin': 'KSN',
    'destination': 'GUW',
    'flight_number': 'HAT170',
    'date': '2024-05-22',
    'price': 883},
   {'origin': 'GUW',
    'destination': 'KSN',
    'flight_number': 'HAT022',
    'date': '2024-05-26',
    'price': 779}],
  'passengers': [{'first_name': 'Дана',
    'last_name': 'Сагиндыкова',
    'dob': '1956-07-07'},
   {'first_name': 'Айсулу', 'last_name': 'Касымова', 'dob': '1967-04-01'},
   {'first_name': 'Айша', 'last_name': 'Кайратова', 'dob': '1970-01-20'}],
  'payment_history': [{'payment_id': 'gift_card_3576581', 'amount': 4986}],
  'created_at': '2024-05-02T03:10:19',
  'total_baggages': 5,
  'nonfree_baggages': 0,
  'insurance': 'no'},
 'VAAOXJ': {'reservation_id': 'VAAOXJ',
  'user_id': 'daulet_kasymov_3206',
  'origin': 'DMB',
  'destination': 'URA',
  'flight_type':

In [71]:
output_file_path = "./tau_bench/envs/airline/data/reservations_kk.json"
with open(output_file_path, "w", encoding="utf-8") as f:
    json.dump(adapted_reservations, f, ensure_ascii=False, indent=4)

In [72]:
# Load the original JSON file
file_path = "./tau_bench/envs/airline/tasks.py"
with open(file_path, "r") as f:
    content = f.read()

In [73]:
import ast

In [74]:
tasks_str = content.split('=', 1)[1].strip()

tasks = ast.literal_eval(tasks_str)

In [28]:
tasks

[{'annotator': 0,
  'user_id': 'mia_li_3668',
  'instruction': 'You are mia_li_3668. You want to fly from New York to Seattle on May 20 (one way). You do not want to fly before 11am est. You want to fly in economy. You prefer direct flights but one stopover also fine. If there are multiple options, you prefer the one with the lowest price. You have 3 baggages. You do not want insurance. You want to use your two certificates to pay. If only one certificate can be used, you prefer using the larger one, and pay the rest with your 7447 card. You are reactive to the agent and will not say anything that is not asked. Your birthday is in your user profile so you do not prefer to provide it.',
  'actions': [{'name': 'book_reservation',
    'arguments': {'user_id': 'mia_li_3668',
     'origin': 'JFK',
     'destination': 'SEA',
     'flight_type': 'one_way',
     'cabin': 'economy',
     'flights': [{'flight_number': 'HAT136', 'date': '2024-05-20'},
      {'flight_number': 'HAT039', 'date': '20

In [31]:
tasks[0]['annotator']

{'annotator': 0,
 'user_id': 'mia_li_3668',
 'instruction': 'You are mia_li_3668. You want to fly from New York to Seattle on May 20 (one way). You do not want to fly before 11am est. You want to fly in economy. You prefer direct flights but one stopover also fine. If there are multiple options, you prefer the one with the lowest price. You have 3 baggages. You do not want insurance. You want to use your two certificates to pay. If only one certificate can be used, you prefer using the larger one, and pay the rest with your 7447 card. You are reactive to the agent and will not say anything that is not asked. Your birthday is in your user profile so you do not prefer to provide it.',
 'actions': [{'name': 'book_reservation',
   'arguments': {'user_id': 'mia_li_3668',
    'origin': 'JFK',
    'destination': 'SEA',
    'flight_type': 'one_way',
    'cabin': 'economy',
    'flights': [{'flight_number': 'HAT136', 'date': '2024-05-20'},
     {'flight_number': 'HAT039', 'date': '2024-05-20'}]

In [40]:
tasks[0]['actions'][0]['arguments']

{'user_id': 'mia_li_3668',
 'origin': 'JFK',
 'destination': 'SEA',
 'flight_type': 'one_way',
 'cabin': 'economy',
 'flights': [{'flight_number': 'HAT136', 'date': '2024-05-20'},
  {'flight_number': 'HAT039', 'date': '2024-05-20'}],
 'passengers': [{'first_name': 'Mia', 'last_name': 'Li', 'dob': '1990-04-05'}],
 'payment_methods': [{'payment_id': 'certificate_7504069', 'amount': 250},
  {'payment_id': 'credit_card_4421486', 'amount': 5}],
 'total_baggages': 3,
 'nonfree_baggages': 0,
 'insurance': 'no'}

In [171]:
import copy
tasks_kk = copy.deepcopy(tasks)

In [172]:
for task in tasks_kk:
    # Update the user_id if it exists in the mapping
    original_user_id = task.get("user_id")
    if original_user_id in ids_mapping:
        task["user_id"] = ids_mapping[original_user_id]

    # Some tasks have nested actions with an "arguments" dict.
    # We iterate over each action and update 'origin' and 'destination'
    for action in task.get("actions", []):
        arguments = action.get("arguments", {})
        if "user_id" in arguments:
            arguments["user_id"] = ids_mapping[original_user_id]
        # Update 'origin' if it's present and in the mapping
        if "origin" in arguments:
            arguments["origin"] = city_codes_mapping[arguments["origin"]]

        # Update 'destination' if it's present and in the mapping
        if "destination" in arguments:
            arguments["destination"] = city_codes_mapping[arguments["destination"]]
        if "passengers" in arguments:
            for passenger in arguments.get("passengers", []):
                if (original_user_id, passenger["first_name"], passenger["last_name"]) in user_info_mapping:
                    name = user_info_mapping.get((original_user_id, passenger["first_name"], passenger["last_name"]))
                    passenger["first_name"] = name[1]
                    passenger["last_name"] = name[2]
                else:
                    d_name = passenger_mapping.get((passenger['first_name'], passenger['last_name'], passenger['dob']))
                    if d_name is not None:

                        passenger["first_name"] = d_name["first_name"]
                        passenger["last_name"] = d_name["last_name"]
                    else:
                        rand_passenger = get_random_passenger_dict()
                        passenger["first_name"] = rand_passenger["first_name"]
                        passenger["last_name"] = rand_passenger["last_name"]

In [118]:
user_info_mapping.get(("Mia", "Li"))[0]

'Алексей'

In [85]:
passenger_mapping[('Amelia', 'Ahmed', '1957-03-21')]

{'first_name': 'Жанар', 'last_name': 'Мустафина', 'dob': '1957-03-21'}

In [82]:
passenger_mapping

{('Amelia', 'Ahmed', '1957-03-21'): {'first_name': 'Жанар',
  'last_name': 'Мустафина',
  'dob': '1957-03-21'},
 ('Anya', 'Anderson', '1959-05-28'): {'first_name': 'Данияр',
  'last_name': 'Кайратов',
  'dob': '1959-05-28'},
 ('Mohamed', 'Johnson', '1981-07-16'): {'first_name': 'Айдос',
  'last_name': 'Федоров',
  'dob': '1981-07-16'},
 ('Olivia', 'Sanchez', '1983-05-09'): {'first_name': 'Ерасыл',
  'last_name': 'Васильев',
  'dob': '1983-05-09'},
 ('Liam', 'Nguyen', '1960-08-07'): {'first_name': 'Максат',
  'last_name': 'Касымов',
  'dob': '1960-08-07'},
 ('Harper', 'Taylor', '1986-02-14'): {'first_name': 'Рустам',
  'last_name': 'Айдосов',
  'dob': '1986-02-14'},
 ('Lucas', 'Davis', '1987-10-27'): {'first_name': 'Мадина',
  'last_name': 'Тлеубекова',
  'dob': '1987-10-27'},
 ('Mason', 'Khan', '1983-09-06'): {'first_name': 'Ажар',
  'last_name': 'Касымова',
  'dob': '1983-09-06'},
 ('Omar', 'Davis', '1991-03-04'): {'first_name': 'Ботагоз',
  'last_name': 'Касымова',
  'dob': '1991-03-

In [86]:
passenger_mapping[('Mia', 'Li','1990-04-05')]

KeyError: ('Mia', 'Li', '1990-04-05')

In [173]:
tasks_kk

[{'annotator': 0,
  'user_id': 'aiana_ergazy_3668',
  'instruction': 'You are mia_li_3668. You want to fly from New York to Seattle on May 20 (one way). You do not want to fly before 11am est. You want to fly in economy. You prefer direct flights but one stopover also fine. If there are multiple options, you prefer the one with the lowest price. You have 3 baggages. You do not want insurance. You want to use your two certificates to pay. If only one certificate can be used, you prefer using the larger one, and pay the rest with your 7447 card. You are reactive to the agent and will not say anything that is not asked. Your birthday is in your user profile so you do not prefer to provide it.',
  'actions': [{'name': 'book_reservation',
    'arguments': {'user_id': 'aiana_ergazy_3668',
     'origin': 'ALA',
     'destination': 'PPK',
     'flight_type': 'one_way',
     'cabin': 'economy',
     'flights': [{'flight_number': 'HAT136', 'date': '2024-05-20'},
      {'flight_number': 'HAT039',

In [117]:
tasks

[{'annotator': 0,
  'user_id': 'mia_li_3668',
  'instruction': 'You are mia_li_3668. You want to fly from New York to Seattle on May 20 (one way). You do not want to fly before 11am est. You want to fly in economy. You prefer direct flights but one stopover also fine. If there are multiple options, you prefer the one with the lowest price. You have 3 baggages. You do not want insurance. You want to use your two certificates to pay. If only one certificate can be used, you prefer using the larger one, and pay the rest with your 7447 card. You are reactive to the agent and will not say anything that is not asked. Your birthday is in your user profile so you do not prefer to provide it.',
  'actions': [{'name': 'book_reservation',
    'arguments': {'user_id': 'mia_li_3668',
     'origin': 'JFK',
     'destination': 'SEA',
     'flight_type': 'one_way',
     'cabin': 'economy',
     'flights': [{'flight_number': 'HAT136', 'date': '2024-05-20'},
      {'flight_number': 'HAT039', 'date': '20